# Estimate Telework Duration Model

This notebook illustrates how to re-estimate a single model component for ActivitySim.  This process 
includes running ActivitySim in estimation mode to read household travel survey files and write out
the estimation data bundles used in this notebook.  To review how to do so, please visit the other
notebooks in this directory.

## Load libraries

In [1]:
import os
import larch
import pandas as pd
import activitysim

from activitysim.estimation.larch import P, component_model, update_coefficients

JAX not found. Some functionality will be unavailable.


## Load estimation data bundle and prepare model for estimation

In [ ]:
modelname = "telework_duration"

from activitysim.estimation.larch import component_model

model, data = component_model(
    modelname,
    edb_directory=f"output-est-mode/estimation_data_bundle/{modelname}/",
    return_data=True,
)

## Review data loaded from the EDB

The next step is to read the EDB, including the coefficients, model settings, utilities specification, and chooser and alternative data.

### Coefficients

In [3]:
data.coefficients

,value,constrain
coefficient_name,,
coef_unavailable,-999.0,F
coef_acs_0_2h,-0.2,F
coef_acs_2_4h,-0.1,F
coef_acs_8_plus_h,-0.4,F
coef_parttime_2_4h,-0.3,F
coef_parttime_4_8h,-0.3,F
coef_parttime_8_plus_h,-0.4,F


### Utility Specification

In [4]:
data.spec

,Label,Description,Expression,3-5 hours,5-7 hours,7-9 hours,9+ hours
0,util_asc,alternative specific constant,@1,coef_acs_3_5,coef_acs_5_7,NaN,coef_asc_9_plus
1,util_part_time,part-time worker,@df.ptype==2,coef_part_time_3_5,coef_part_time_5_7,NaN,coef_part_time_9_plus
2,util_university_student,university student,@df.ptype==3,coef_university_student_3_5,coef_university_student_5_7,NaN,NaN
3,util_work_from_home,work from home,work_from_home,NaN,NaN,NaN,coef_work_from_home_9_plus
4,util_hybrid,hybrid worker,(cdap_activity=='M')&(ptype<3),NaN,NaN,coef_hybrid_7_plus,coef_hybrid_7_plus
5,util_income_150_plus,income 150k+,(income_in_thousands>=150),coef_income_150_plus_3_7,coef_income_150_plus_3_7,NaN,NaN
6,util_workplace_mc_logsum,workplace mode choice logsum,"@np.where(df.workplace_zone_id > -1, df.workpl...",coef_workplace_mc_logsum_3_7,coef_workplace_mc_logsum_3_7,NaN,NaN
7,util_dummy_missing_workplace,util_dummy_missing_workplace,@np.where((df.workplace_zone_id<=-1)&(~df.work...,coef_dummy_missing_workplace_3_5,coef_dummy_missing_workplace_5_7,NaN,NaN
8,util_intrazonal,intrazonal,home_zone_id==workplace_zone_id,coef_intrazonal_3_5,NaN,NaN,NaN


### Chooser Data

In [5]:
data.chooser_data

,person_id,model_choice,override_choice,util_part_time,PNUM,age,sex,pemploy,pstudent,is_student,...,workplace_location_accessibility_0,workplace_location_accessibility_1,workplace_location_accessibility_2,othdiscr_accessibility_0,othdiscr_accessibility_1,othdiscr_accessibility_2,shopping_accessibility_0,shopping_accessibility_1,shopping_accessibility_2,override_choice_code
household_id,,,,,,,,,,,,,,,,,,,,,
43394,85660,9+ hours,5-7 hours,0.0,1,27,1,1,3,False,...,12.439093,11.081183,11.010344,15.741796,14.071562,14.179500,12.927407,10.816278,11.013279,2
43399,85668,5-7 hours,3-5 hours,1.0,1,35,2,2,3,False,...,12.413072,11.175996,11.001979,15.788809,13.624172,13.873905,13.294888,11.209387,10.879597,1
43400,85670,3-5 hours,7-9 hours,0.0,1,45,2,1,2,True,...,11.794772,10.544920,10.708367,15.651811,14.028888,14.597265,11.898473,9.356192,9.520027,3
43400,85671,9+ hours,7-9 hours,0.0,2,52,1,1,3,False,...,11.794772,10.544920,10.708367,15.651811,14.028888,14.597265,11.898473,9.356192,9.520027,3
43402,85674,9+ hours,7-9 hours,0.0,1,47,1,1,3,False,...,11.444002,10.175991,10.104947,14.271339,13.640185,13.749169,9.686552,8.310021,8.198408,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49748,96856,3-5 hours,7-9 hours,0.0,1,19,2,1,3,False,...,12.134927,10.839010,10.701627,17.340882,15.934239,16.487026,14.251325,12.542380,11.811642,3
49751,96862,3-5 hours,7-9 hours,0.0,1,49,2,1,3,False,...,11.718314,10.772296,10.523315,15.271568,12.832681,12.799794,11.675215,9.561950,9.878896,3
49751,96865,3-5 hours,3-5 hours,1.0,4,45,2,2,3,False,...,11.718314,10.772296,10.523315,15.271568,12.832681,12.799794,11.675215,9.561950,9.878896,1


## Estimate

With the model setup for estimation, the next step is to estimate the model coefficients.  Make sure to use a sufficiently large enough household sample and set of zones to avoid an over-specified model, which does not have a numerically stable likelihood maximizing solution.  Larch has a built-in estimation methods including BHHH, and also offers access to more advanced general purpose non-linear optimizers in the `scipy` package, including SLSQP, which allows for bounds and constraints on parameters.  BHHH is the default and typically runs faster, but does not follow constraints on parameters.

In [6]:
model.estimate(method='SLSQP',maxiter=500)

,value,best,initvalue,minimum,maximum,nullvalue,holdfast
param_name,,,,,,,
coef_acs_3_5,-1.713574,-1.713574,0.0,-50.0,50.0,0.0,0
coef_acs_5_7,-1.798224,-1.798224,0.0,-50.0,50.0,0.0,0
coef_asc_9_plus,-1.530817,-1.530817,0.0,-50.0,50.0,0.0,0
coef_dummy_missing_workplace_3_5,1.699246,1.699246,0.0,-50.0,50.0,0.0,0
coef_dummy_missing_workplace_5_7,1.284953,1.284953,0.0,-50.0,50.0,0.0,0
coef_hybrid_7_plus,-6.697958,-6.697958,0.0,-50.0,50.0,0.0,0
coef_income_150_plus_3_7,-0.194289,-0.194289,0.0,-50.0,50.0,0.0,0
coef_intrazonal_3_5,0.495109,0.495109,0.0,-50.0,50.0,0.0,0
coef_part_time_3_5,1.997151,1.997151,0.0,-50.0,50.0,0.0,0


┣                x: coef_acs_3_5                       -1.713574
┃                   coef_acs_5_7                       -1.798224
┃                   coef_asc_9_plus                    -1.530817
┃                   coef_dummy_missing_workplace_3_5    1.699246
┃                   coef_dummy_missing_workplace_5_7    1.284953
┃                   coef_hybrid_7_plus                 -6.697958
┃                   coef_income_150_plus_3_7           -0.194289
┃                   coef_intrazonal_3_5                 0.495109
┃                   coef_part_time_3_5                  1.997151
┃                   coef_part_time_5_7                  1.849555
┃                   coef_part_time_9_plus              -1.644802
┃                   coef_university_student_3_5         2.160429
┃                   coef_university_student_5_7         1.848575
┃                   coef_work_from_home_9_plus         -0.264049
┃                   coef_workplace_mc_logsum_3_7        0.062393
┃                   dtype: float64
┣          logloss: np.float64(1.0053578779641739)
┣        d_logloss: coef_acs_3_5                        0.000009
┃                   coef_acs_5_7                        0.000062
┃                   coef_asc_9_plus                     0.000019
┃                   coef_dummy_missing_workplace_3_5   -0.000086
┃                   coef_dummy_missing_workplace_5_7   -0.000175
┃                   coef_hybrid_7_plus                 -0.000208
┃                   coef_income_150_plus_3_7           -0.000050
┃                   coef_intrazonal_3_5                 0.000282
┃                   coef_part_time_3_5                 -0.000075
┃                   coef_part_time_5_7                 -0.000181
┃                   coef_part_time_9_plus              -0.000164
┃                   coef_university_student_3_5        -0.000167
┃                   coef_university_student_5_7        -0.000243
┃                   coef_work_from_home_9_plus         -0.000062
┃                   coef_workplace_mc_logsum_3_7       -0.000024
┃                   dtype: float64
┣              nit: 57
┣             nfev: 57
┣             njev: 57
┣           status: 0
┣          message: 'Optimization terminated successfully'
┣          success: np.True_
┣     elapsed_time: datetime.timedelta(microseconds=388796)
┣           method: 'SLSQP'
┣          n_cases: 2310
┣ iteration_number: 57
┣          loglike: np.float64(-2322.376698097242)

In [7]:
model.estimation_statistics()

<xmle.Elem 'div' with 1 children>

### Estimated coefficients

In [8]:
model.parameter_summary()

,Value,Std Err,t Stat,Signif,Null Value
Parameter,,,,,
coef_acs_3_5,-1.71,0.0956,-17.92,***,0.00
coef_acs_5_7,-1.80,0.0950,-18.94,***,0.00
coef_asc_9_plus,-1.53,0.0884,-17.31,***,0.00
coef_dummy_missing_workplace_3_5,1.70,0.202,8.42,***,0.00
coef_dummy_missing_workplace_5_7,1.28,0.227,5.67,***,0.00
coef_hybrid_7_plus,-6.70,1.45,-4.62,***,0.00
coef_income_150_plus_3_7,-0.194,0.114,-1.70,,0.00
coef_intrazonal_3_5,0.495,0.232,2.13,*,0.00
coef_part_time_3_5,2.00,0.227,8.81,***,0.00


## Output Estimation Results

In [ ]:
from activitysim.estimation.larch import update_coefficients

result_dir = data.edb_directory / "estimated"
update_coefficients(
    model,
    data,
    result_dir,
    output_file=f"{modelname}_coefficients_revised.csv",
)

### Write the model estimation report, including coefficient t-statistic and log likelihood

In [ ]:
model.to_xlsx(
    result_dir/f"telework_duration_model_estimation.xlsx", 
    data_statistics=True,
)

## Next Steps

The final step is to either manually or automatically copy the `*_coefficients_revised.csv` file to the configs folder, rename it to `*_coefficients.csv`, and run ActivitySim in simulation mode.

In [ ]:
pd.read_csv(result_dir / f"{modelname}_coefficients_revised.csv")